In [1]:
import time
import re
import pytz
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import selenium.common.exceptions as se

amazon_base = "https://www.amazon.com.tr"
amazon_login_page = "https://www.amazon.com.tr/ap/signin?openid.pape.max_auth_age=0&openid.return_to=https%3A%2F%2Fwww.amazon.com.tr%2F%3Fref_%3Dnav_ya_signin&openid.identity=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.assoc_handle=trflex&openid.mode=checkid_setup&openid.claimed_id=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.ns=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0"

TR_MONTHS = {
    "Ocak": 1, "Şubat": 2, "Mart": 3, "Nisan": 4, "Mayıs": 5, "Haziran": 6,
    "Temmuz": 7, "Ağustos": 8, "Eylül": 9, "Ekim": 10, "Kasım": 11, "Aralık": 12
}

def make_driver(detach=True, headless=False):
    chrome_options = webdriver.ChromeOptions()
    if detach:
        chrome_options.add_experimental_option("detach", True)
    if headless:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1400,900")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver

In [2]:
def build_review_url(asin: str, page: int) -> str:
    ref = "cm_cr_arp_d_viewopt_sr" if page <= 1 else f"cm_cr_arp_d_paging_btm_next_{page}"
    return (
        f"{amazon_base}/product-reviews/{asin}/ref={ref}"
        f"?ie=UTF8&reviewerType=all_reviews&sortBy=recent&pageNumber={page}"
    )

def build_product_url(asin: str) -> str:
    return f"{amazon_base}/dp/{asin}"

In [3]:
def wait_reviews_stable(driver, timeout=20, stable_rounds=3, poll=0.25):
    end = time.time() + timeout
    last = -1
    stable = 0
    while time.time() < end:
        cards = driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]')
        n = len(cards)
        if n == last and n > 0:
            stable += 1
            if stable >= stable_rounds:
                return n
        else:
            stable = 0
            last = n
        time.sleep(poll)
    return len(driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]'))


In [4]:
def safe_text(el) -> str:
    try:
        return el.text.strip()
    except Exception:
        return ""

def parse_amazon_tr_rating(text: str):
    if not text:
        return None
    m = re.search(r"üzerinden\s+(\d+(?:[.,]\d+)?)", text)
    if not m:
        m = re.search(r"(\d+(?:[.,]\d+)?)", text)
    if not m:
        return None
    return float(m.group(1).replace(",", "."))


def parse_amazon_tr_date_to_iso(date_text: str):
    if not date_text:
        return None

    m = re.search(r"(\d{1,2})\s+([A-Za-zÇĞİÖŞÜçğıöşü]+)\s+(\d{4})", date_text)
    if not m:
        return None

    day = int(m.group(1))
    mon_name = m.group(2).strip()
    year = int(m.group(3))
    mon = TR_MONTHS.get(mon_name)
    if not mon:
        return None
    return datetime(year, mon, day).date().isoformat()

In [5]:
def get_product_meta(driver, asin: str, wait_timeout: int = 15) -> dict:
    wait = WebDriverWait(driver, wait_timeout)

    product_url = build_product_url(asin)
    driver.get(product_url)

    product_title = None
    try:
        title_el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#productTitle")))
        product_title = title_el.text.strip()
    except Exception:
        product_title = None

    product_description = None
    desc_candidates = [
        "#feature-bullets ul li span.a-list-item",
        "ul.a-unordered-list.a-vertical li span.a-list-item",
        "#productDescription",
    ]

    bullets = []
    for sel in desc_candidates:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            texts = [e.text.strip() for e in els if e.text and e.text.strip()]
            if texts:
                if sel == "#productDescription" and not bullets:
                    product_description = " ".join(texts).strip()
                    break
                else:
                    bullets.extend(texts)
        except Exception:
            continue

    if bullets:
        product_description = " | ".join(dict.fromkeys(bullets))

    return {
        "product_title": product_title,
        "product_description": product_description,
        "product_url": product_url
    }

In [6]:
def find_local_and_global_cards(driver):
    container = driver.find_element(By.CSS_SELECTOR, "#cm_cr-review_list")

    cards = container.find_elements(By.CSS_SELECTOR, 'li[data-hook="review"], div[data-hook="review"]')

    local_headers = container.find_elements(By.CSS_SELECTOR, 'h3[data-hook="arp-local-reviews-header"]')
    local_text = local_headers[0].text.strip() if local_headers else ""

    is_local_section = ("Ülke:" in local_text) or ("Türkiye" in local_text)

    if is_local_section:
        return cards, []
    else:
        return [], cards

In [8]:
def extract_rating_from_card_text(card) -> float | None:
    try:
        txt = card.get_attribute("innerText") or card.get_attribute("textContent") or ""
        txt = txt.replace("\u00a0", " ").strip()  # nbsp temizle
    except Exception:
        txt = (card.text or "").strip()

    if not txt:
        return None

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*yıldız\s*üzerinden\s*(\d+(?:[.,]\d+)?)",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(2).replace(",", "."))

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*üzerinden\s*(\d+(?:[.,]\d+)?)",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(2).replace(",", "."))

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*yıldız",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(1).replace(",", "."))

    return None

In [9]:
def parse_review_card_to_row(card, asin: str, current_reviews_url: str, product_meta: dict = None):
    # Title
    title = None
    try:
        title_el = card.find_element(By.CSS_SELECTOR, '[data-hook="review-title"]')
        title = safe_text(title_el)
    except Exception:
        pass

    # Rating -> float
    rating_val = extract_rating_from_card_text(card)

    # Body
    body = None
    try:
        body_el = card.find_element(By.CSS_SELECTOR, '[data-hook="review-body"]')
        body = safe_text(body_el).replace("\n", " ").strip()
    except Exception:
        pass

    # Date -> ISO
    review_date_iso = None
    try:
        date_el = card.find_element(By.CSS_SELECTOR, 'span[data-hook="review-date"]')
        review_date_iso = parse_amazon_tr_date_to_iso(safe_text(date_el))
    except Exception:
        pass

    review_id = None
    try:
        review_id = card.get_attribute("id")  # R32D2E5N2T6S02
    except Exception:
        pass

    turkey_tz = pytz.timezone("Europe/Istanbul")
    collected_at = datetime.now(turkey_tz).strftime("%Y-%m-%d %H:%M")

    return {
            "review_id": review_id,
            "asin": asin,
            "title": title,
            "rating": rating_val,
            "body": body,
            "review_date": review_date_iso,
    
            # product meta
            "product_title": (product_meta or {}).get("product_title"),
            "product_description": (product_meta or {}).get("product_description"),
    
            "product_url": build_product_url(asin),
            "source": "amazon_tr",
            "review_summary": None,
            "collected_at": collected_at,
    }

In [10]:
def scrape_most_recent_local_reviews_df(
    driver,
    asin: str,
    target_n: int = 80,
    max_pages: int = 80,
    wait_timeout: int = 15
):
    wait = WebDriverWait(driver, wait_timeout)
    seen = set()
    rows = []

    # --- product meta (visit product page once) ---
    try:
        product_meta = get_product_meta(driver, asin, wait_timeout=wait_timeout)
    except Exception:
        product_meta = {"product_title": None, "product_description": None}

    driver.get(build_review_url(asin, 1))

    last_url = None

    for page in range(1, max_pages + 1):
        # --- wait for reviews to load ---
        try:
            # container gelsin
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#cm_cr-review_list")))
    
            # kartlar DOM'da var olsun (en az 1 tane)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]')))
    
            # VE EN ÖNEMLİSİ: kart sayısı artık değişmiyor olsun
            wait_reviews_stable(driver, timeout=25, stable_rounds=4)

        except se.TimeoutException:
            break


        # --- prevent infinite loop if Amazon keeps us on same page ---
        if driver.current_url == last_url:
            # page did not change after clicking next
            break
        last_url = driver.current_url

        # --- get review cards (local vs global) ---
        local_cards, global_cards = find_local_and_global_cards(driver)

        # stop if global started or locals ended
        if len(global_cards) > 0 or len(local_cards) == 0:
            break

        # --- parse cards ---
        for i, card in enumerate(local_cards):
            try:
                rid = card.get_attribute("id")
            except Exception:
                rid = None
        
            if rid and rid in seen:
                continue
        
            try:
                row = parse_review_card_to_row(card, asin, driver.current_url, product_meta=product_meta)
            except se.StaleElementReferenceException:
                cards2 = driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list li[data-hook="review"], #cm_cr-review_list div[data-hook="review"]')
                if i >= len(cards2):
                    continue
                row = parse_review_card_to_row(cards2[i], asin, driver.current_url, product_meta=product_meta)
                rid = row.get("review_id")
        
            if rid:
                seen.add(rid)
            rows.append(row)

            if len(seen) >= target_n:
                df = pd.DataFrame(rows)
                df = df.drop_duplicates(subset=["review_id"])
                return df.head(target_n)

        time.sleep(0.6)

        # --- click NEXT (Sonraki) instead of building URL ---
        try:
            # If next is disabled, stop
            disabled_next = driver.find_elements(By.CSS_SELECTOR, "li.a-disabled.a-last")
            if disabled_next:
                break

            next_a = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.a-last a")))
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_a)

            # click normally, fallback to JS click (overlays happen a lot)
            try:
                next_a.click()
            except Exception:
                driver.execute_script("arguments[0].click();", next_a)

        except se.TimeoutException:
            # no next button => last page
            break

    df = pd.DataFrame(rows[:target_n])

    if "review_id" in df.columns:
        df = df.drop_duplicates(subset=["review_id"])
    else:
        df = df.drop_duplicates()
    df = pd.DataFrame(rows)

    return df


In [11]:
def scrape_many_asins(driver, asins, target_n=80, max_pages=80):
    all_dfs = []
    for idx, asin in enumerate(asins, 1):
        try:
            print(f"[{idx}/{len(asins)}] Scraping ASIN: {asin}")
            df = scrape_most_recent_local_reviews_df(
                driver,
                asin=asin,
                target_n=target_n,
                max_pages=max_pages
            )
            print(f"  -> got {len(df)} reviews")
            all_dfs.append(df)
        except Exception as e:
            print(f"  !! FAILED ASIN {asin}: {type(e).__name__}: {e}")
            continue

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    return pd.DataFrame(columns=[
        "asin",
        "title","rating","body","review_date",
        "product_title","product_description",
        "product_url","source","review_summary","collected_at"
    ])

In [13]:
driver = make_driver(detach=True, headless=False)

# Login (manuel)
driver.get(amazon_login_page)

In [14]:
asins = ["B07RFD8YZJ", "B0BP2SVYQS", "B0BC6LGNFT"]

df_all = scrape_many_asins(driver, asins, target_n=80, max_pages=80)
df_all = df_all.drop("review_id", axis=1)
#df_all.to_csv("amazon_tr_reviews_local_only.csv", index=False, encoding="utf-8-sig")
df_all.head()

[1/3] Scraping ASIN: B07RFD8YZJ
  -> got 80 reviews
[2/3] Scraping ASIN: B0BP2SVYQS
  -> got 80 reviews
[3/3] Scraping ASIN: B0BC6LGNFT
  -> got 80 reviews


AttributeError: 'list' object has no attribute 'drop'